<a href="https://colab.research.google.com/github/vipin-jangra/face-age-estimation-CNN/blob/main/combinedData_CNN9_A3_FT1_updated_results.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import cv2
import keras
import os
from zipfile import ZipFile
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
import tensorflow as tf
from keras.applications.efficientnet import EfficientNetB0
from keras.models import Model
from keras.layers import Dense
from keras.layers import Conv2D, AveragePooling2D, GlobalAveragePooling2D
from keras.callbacks import EarlyStopping, ModelCheckpoint
from sklearn.model_selection import train_test_split
from keras.preprocessing.image import load_img, img_to_array
from keras.optimizers import Adam
from keras.utils import to_categorical
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from keras.applications.resnet50 import preprocess_input


In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


Data Preparation

In [ ]:
 #Unzipping the dataset file combined_faces.zip

combined_faces_zip_path = "/content/drive/My Drive/Dataset/Dataset2/combinedData.zip"
output_directory = "/content/combinedData"
# Create the output directory if it doesn't exist
os.makedirs(output_directory, exist_ok=True)

with ZipFile(combined_faces_zip_path, 'r') as myzip:
    myzip.extractall(output_directory)
    print('Done unzipping combinedData.zip')

Done unzipping combinedData.zip


In [ ]:
import pandas as pd
train_df = pd.read_csv( "/content/combinedData/combinedData/train_labels.csv")
test_df = pd.read_csv("/content/combinedData/combinedData/test_labels.csv")


In [ ]:
train_df.head()

,filepath,age
0,/content/combinedData/combinedData/train/age_1...,10
1,/content/combinedData/combinedData/train/age_2...,26
2,/content/combinedData/combinedData/train/age_7...,72
3,/content/combinedData/combinedData/train/age_3...,33
4,/content/combinedData/combinedData/train/age_7...,75


In [ ]:
test_df.head()

,filepath,age
0,/content/combinedData/combinedData/test/age_66...,66
1,/content/combinedData/combinedData/test/age_17...,17
2,/content/combinedData/combinedData/test/age_3_...,3
3,/content/combinedData/combinedData/test/age_55...,55
4,/content/combinedData/combinedData/test/age_8_...,8


In [ ]:
# Define age ranges
age_ranges = [(1, 2), (3, 9), (10, 20), (21, 27), (28, 45), (46, 65), (66, 80)]

In [ ]:
def categorize_age(age):
    for range_name, (start, end) in enumerate(age_ranges):
        if start <= age <= end:
            return range_name
    return None

In [ ]:
train_df['target'] = train_df['age'].map(categorize_age)
test_df['target'] = test_df['age'].map(categorize_age)

In [ ]:
train_df.head()

,filepath,age,target
0,/content/combinedData/combinedData/train/age_1...,10,2
1,/content/combinedData/combinedData/train/age_2...,26,3
2,/content/combinedData/combinedData/train/age_7...,72,6
3,/content/combinedData/combinedData/train/age_3...,33,4
4,/content/combinedData/combinedData/train/age_7...,75,6


In [ ]:
test_df.head()

,filepath,age,target
0,/content/combinedData/combinedData/test/age_66...,66,6
1,/content/combinedData/combinedData/test/age_17...,17,2
2,/content/combinedData/combinedData/test/age_3_...,3,1
3,/content/combinedData/combinedData/test/age_55...,55,5
4,/content/combinedData/combinedData/test/age_8_...,8,1


In [ ]:
train_df.shape

(8205, 3)

In [ ]:
test_df.shape

(2052, 3)

In [ ]:
# Converting the filenames and target class labels into lists for augmented train and test datasets.

train_filenames_list = list(train_df['filepath'])
train_labels_list = list(train_df['age'])

test_filenames_list = list(test_df['filepath'])
test_labels_list = list(test_df['age'])

In [ ]:
# Creating tensorflow constants of filenames and labels for augmented train and test datasets from the lists defined above.

train_aug_filenames_tensor = tf.constant(train_filenames_list)
train_aug_labels_tensor = tf.constant(train_labels_list)

test_filenames_tensor = tf.constant(test_filenames_list)
test_labels_tensor = tf.constant(test_labels_list)

In [ ]:
# Defining a function to read the image, decode the image from given tensor and one-hot encode the image label class.
# Changing the channels para in tf.io.decode_jpeg from 3 to 1 changes the output images from RGB coloured to grayscale.

num_classes = 7

def _parse_function(filename, label):

    image_string = tf.io.read_file(filename)
    image_decoded = tf.io.decode_jpeg(image_string, channels=3)    # channels=1 to convert to grayscale, channels=3 to convert to RGB.
    label = tf.one_hot(label, num_classes)

    return image_decoded, label

In [ ]:
# Getting the dataset ready for the neural network.
# Using the tensor vectors defined above, accessing the images in the dataset and passing them through the function defined above.

train_aug_dataset = tf.data.Dataset.from_tensor_slices((train_aug_filenames_tensor, train_aug_labels_tensor))
train_aug_dataset = train_aug_dataset.map(_parse_function)
# train_aug_dataset = train_aug_dataset.repeat(3)
train_aug_dataset = train_aug_dataset.batch(32)    # Same as batch_size hyperparameter in model.fit() below.

test_dataset = tf.data.Dataset.from_tensor_slices((test_filenames_tensor, test_labels_tensor))
test_dataset = test_dataset.map(_parse_function)
# test_dataset = test_dataset.repeat(3)
test_dataset = test_dataset.batch(32)    # Same as batch_size hyperparameter in model.fit() below.

In [ ]:
# Load the pre-trained ResNet50 model
base_model = EfficientNetB0(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

# Add custom layers on top of the base model
x = base_model.output
x = GlobalAveragePooling2D()(x)

x = Dense(132, activation='relu')(x)
predictions = Dense(len(age_ranges), activation='softmax')(x)

# Create the complete model
model = Model(inputs=base_model.input, outputs=predictions)

# Compile the model
model.compile(optimizer=Adam(learning_rate=0.0001), loss='categorical_crossentropy', metrics=['accuracy'])

16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [ ]:
early_stopping = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)
model_checkpoint = ModelCheckpoint(
    '/content/drive/MyDrive/Dataset/Dataset2/model-weights/combinedData_CNN1_A3_FT1.keras',  # Filepath to save the best model
    monitor='val_accuracy',  # Metric to monitor
    verbose=1,
    save_best_only=True,  # Only save the best model
    mode='max'  # Mode to determine best (maximizing validation accuracy)
)

In [ ]:
# Train your model
history = model.fit(train_aug_dataset, validation_data=test_dataset, epochs=60,batch_size=32, callbacks=[early_stopping,model_checkpoint])

Epoch 1/60
257/257 ━━━━━━━━━━━━━━━━━━━━ 0s 289ms/step - accuracy: 0.2373 - loss: 0.1490
Epoch 1: val_accuracy improved from -inf to 0.89961, saving model to /content/drive/MyDrive/Dataset/Dataset2/model-weights/combinedData_CNN1_A3_FT1.keras
257/257 ━━━━━━━━━━━━━━━━━━━━ 157s 344ms/step - accuracy: 0.2373 - loss: 0.1490 - val_accuracy: 0.8996 - val_loss: 0.1606
Epoch 2/60
257/257 ━━━━━━━━━━━━━━━━━━━━ 0s 125ms/step - accuracy: 0.4659 - loss: 0.1651
Epoch 2: val_accuracy did not improve from 0.89961
257/257 ━━━━━━━━━━━━━━━━━━━━ 57s 133ms/step - accuracy: 0.4659 - loss: 0.1651 - val_accuracy: 0.5858 - val_loss: 0.1669
Epoch 3/60
257/257 ━━━━━━━━━━━━━━━━━━━━ 0s 127ms/step - accuracy: 0.4660 - loss: 0.1659
Epoch 3: val_accuracy did not improve from 0.89961
257/257 ━━━━━━━━━━━━━━━━━━━━ 35s 137ms/step - accuracy: 0.4660 - loss: 0.1658 - val_accuracy: 0.7149 - val_loss: 0.1669
Epoch 4/60
257/257 ━━━━━━━━━━━━━━━━━━━━ 0s 126ms/step - accuracy: 0.5176 - loss: 0.1667
Epoch 4: val_accuracy did not i

In [ ]:
# Save the training history
history_dict = history.history

# Plot the training and validation metrics
acc = history_dict['accuracy']
val_acc = history_dict['val_accuracy']
loss = history_dict['loss']
val_loss = history_dict['val_loss']

In [ ]:
epochs = range(1, len(acc) + 1)

# Plot accuracy
plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plt.plot(epochs, acc, 'bo', label='Training accuracy')
plt.plot(epochs, val_acc, 'b', label='Validation accuracy')
plt.title('Training and validation accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()

In [ ]:
# Plot loss
plt.subplot(1, 2, 2)
plt.plot(epochs, loss, 'bo', label='Training loss')
plt.plot(epochs, val_loss, 'b', label='Validation loss')
plt.title('Training and validation loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()

In [ ]:
# Load the best model
model.load_weights('/content/drive/MyDrive/Dataset/Dataset2/model-weights/combinedData_CNN1_A3_FT1.keras')
# Evaluate the model
# Evaluate the model on the validation dataset
val_loss, val_accuracy = model.evaluate(test_dataset, verbose=1)

print(f"Validation Loss: {val_loss}")
print(f"Validation Accuracy: {val_accuracy}")


/usr/local/lib/python3.10/dist-packages/keras/src/saving/saving_lib.py:576: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 432 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


65/65 ━━━━━━━━━━━━━━━━━━━━ 190s 3s/step - accuracy: 0.9250 - loss: 0.1615
Validation Loss: 0.16000579297542572
Validation Accuracy: 0.9244639277458191


In [ ]:
representative_ages = [np.mean(rng) for rng in age_ranges]

In [ ]:
# Convert predictions and true labels back to representative ages
y_pred = model.predict(test_dataset)
print(y_pred)
y_pred_ages = [representative_ages[np.argmax(pred)] for pred in y_pred]

# Since val_labels is one-hot encoded, convert it to a numpy array and then to representative ages
val_labels_np = np.concatenate([y for x, y in test_dataset], axis=0)
y_true_ages = [representative_ages[np.argmax(true)] for true in val_labels_np]

# Calculate MAE
mae = np.mean(np.abs(np.array(y_true_ages) - np.array(y_pred_ages)))
print(f'Test MAE: {mae:.2f}')

65/65 ━━━━━━━━━━━━━━━━━━━━ 175s 3s/step
Test MAE: 2.51


In [ ]:
# Calculate classification metrics
accuracy = accuracy_score(true_classes, predicted_classes)
precision = precision_score(true_classes, predicted_classes, average='weighted')
recall = recall_score(true_classes, predicted_classes, average='weighted')
f1 = f1_score(true_classes, predicted_classes, average='weighted')

print(f"Accuracy: {accuracy:.2f}")
print(f"Precision: {precision:.2f}")
print(f"Recall: {recall:.2f}")
print(f"F1-score: {f1:.2f}")

Accuracy: 0.92
Precision: 0.85
Recall: 0.92
F1-score: 0.89


/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
